# LangChain Integration

> **Time:** ~15 minutes | **Level:** Intermediate

Make a **LangChain agent self-improve** using Reflexio. This notebook covers five integration components:

1. **`ReflexioCallbackHandler`** — captures conversations and publishes them to Reflexio automatically
2. **`get_reflexio_context`** — searches Reflexio for context relevant to a specific query
3. **`ReflexioSearchTool`** — lets agents query Reflexio mid-conversation for guidance
4. **`ReflexioRetriever`** — LangChain `BaseRetriever` for RAG-style playbook/profile injection
5. **`ReflexioChatModel`** — wraps any ChatModel to **always** search Reflexio before every LLM call

### Prerequisites

1. A running Reflexio backend (`uv run reflexio services start --only backend`)
2. `langchain-core`, `langchain-openai` (or another chat model), and `langgraph` installed
3. A `.env` file with at least one LLM API key

```bash
# Quote the brackets for zsh compatibility
uv pip install 'reflexio-client[langchain]' langchain-openai langgraph

# Or install dependencies directly for local development
uv pip install langchain-core langchain-openai langgraph
```

## 1. Setup

In [ ]:
import uuid

from _display_helpers import *
from dotenv import load_dotenv

load_dotenv()

from reflexio import ReflexioClient

# Each run uses a unique ID so the notebook is idempotent
RUN_ID = uuid.uuid4().hex[:8]

url, api_key = load_env()
client = ReflexioClient(url_endpoint=url, api_key=api_key)

In [ ]:
# Set stride to 1 so profile and playbook extraction runs on every interaction
config = client.get_config()
config.stride_size = 1
config.window_size = 10
client.set_config(config)
show_success("Config updated: stride_size=1, window_size=10")

In [ ]:
from langchain_openai import ChatOpenAI

# Use any LangChain-compatible chat model
# For Anthropic: from langchain_anthropic import ChatAnthropic
# For Google: from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 2. Capturing Conversations with `ReflexioCallbackHandler`

The callback handler is the core of the integration. Add it to any LangChain chain or agent to automatically capture conversations and publish them to Reflexio for learning.

The handler:
- Captures user messages, assistant responses, and tool calls
- Publishes everything to Reflexio when the chain completes (fire-and-forget)
- Works with any LangChain chain, agent, or runnable

Since we set `stride_size=1` above, Reflexio will run profile and playbook extraction on **every** published interaction.

### Agent conversation with implicit playbook signals

We'll create an Acme Electronics customer support agent with product search and order lookup tools. The agent will handle a two-turn conversation where the customer asks about a product, and then follows up about their order — providing implicit signals that the agent should have been more proactive.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

from reflexio.integrations.langchain import ReflexioCallbackHandler


@tool
def product_search(query: str) -> str:
    """Search the Acme Electronics product catalog."""
    return (
        f"Search results for '{query}':\n"
        "1. Acme ProBook 15 - $1,299 (in stock)\n"
        "   14-core CPU, 32 GB RAM, 1 TB SSD\n"
        "2. Acme ProBook 13 - $999 (in stock)\n"
        "   10-core CPU, 16 GB RAM, 512 GB SSD\n"
        "3. Acme UltraStation Desktop - $2,499 (backordered)\n"
        "   24-core CPU, 64 GB RAM, 2 TB SSD"
    )


@tool
def order_lookup(order_id: str) -> str:
    """Look up order status and details by order ID."""
    return (
        f"Order {order_id}: Acme ProBook 15\n"
        "- Status: Shipped (in transit)\n"
        "- Tracking: 1Z999AA10123456784\n"
        "- Estimated delivery: Tomorrow by 5 PM\n"
        "- Warranty: 2-year standard (expires 2028-03-15)"
    )


# Create a ReAct agent with product support tools
agent = create_react_agent(llm, [product_search, order_lookup])

In [ ]:
# --- Turn 1: Customer asks about products ---
handler = ReflexioCallbackHandler(
    client=client,
    user_id=f"customer_{RUN_ID}",
    agent_version="v1.0",
    session_id=f"langchain_demo_{RUN_ID}",
    source="langchain_notebook",
)

result = agent.invoke(
    {"messages": [("human", "I'm looking for a laptop for software development. What do you have?")]},
    config={"callbacks": [handler]},
)
show_success("Turn 1 complete")
rprint("[bold]User:[/bold] I'm looking for a laptop for software development. What do you have?")
rprint(f"[bold]Agent:[/bold] {result['messages'][-1].content[:300]}")

In [ ]:
# --- Turn 2: Customer follows up about existing order (implicit playbook signal) ---
handler2 = ReflexioCallbackHandler(
    client=client,
    user_id=f"customer_{RUN_ID}",
    agent_version="v1.0",
    session_id=f"langchain_demo_{RUN_ID}",
    source="langchain_notebook",
)

result2 = agent.invoke(
    {
        "messages": result["messages"]
        + [("human", "Actually I already ordered #ACM-7891 last week. Where is it? You should have asked if I had an existing order.")]
    },
    config={"callbacks": [handler2]},
)
show_success("Turn 2 complete")
rprint("[bold]User:[/bold] Actually I already ordered #ACM-7891 last week. Where is it? You should have asked if I had an existing order.")
rprint(f"[bold]Agent:[/bold] {result2['messages'][-1].content[:300]}")

### Verify the interaction was captured and user playbooks were generated

After the agent runs, the conversation should be stored in Reflexio. Since `stride_size=1`, profile and playbook extraction should have triggered automatically. We verify both the interactions and the user playbooks extracted from the implicit correction in Turn 2.

In [ ]:
import time

time.sleep(2)  # Brief wait for async extraction to complete

interactions = client.get_interactions(user_id=f"customer_{RUN_ID}", top_k=20)
show_interactions(interactions.interactions, title="Captured Interactions")

# Verify user playbooks were extracted from the implicit correction
user_pbs = client.get_user_playbooks(playbook_name="default_playbook_extractor")
show_playbooks(user_pbs.user_playbooks, title="User Playbooks (extracted from implicit correction)")

## 3. Per-Query Context with `get_reflexio_context`

For dynamic, per-query enrichment, use `get_reflexio_context`. This searches Reflexio for insights relevant to a specific query.

In [ ]:
from reflexio.integrations.langchain import get_reflexio_context

# Search for context relevant to a specific query
context = get_reflexio_context(
    client,
    query="customer asks about products",
    agent_version="v1.0",
    user_id=f"customer_{RUN_ID}",
    top_k=5,
)

display(Markdown("### Retrieved Context"))
rprint(context or "[dim]No relevant context found yet — publish more conversations first[/dim]")

### Injecting per-query context into a chain

Use `get_reflexio_context` inside a chain by combining it with a `RunnableLambda`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda


def add_reflexio_context(inputs: dict) -> dict:
    """Fetch relevant Reflexio context and add it to the chain inputs."""
    context = get_reflexio_context(
        client,
        query=inputs["input"],
        agent_version="v1.0",
        user_id=f"customer_{RUN_ID}",
    )
    inputs["reflexio_context"] = context or "No additional context available."
    return inputs


# Chain with dynamic context injection
context_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an Acme Electronics customer support agent.\n\nRelevant context from past interactions:\n{reflexio_context}",
        ),
        ("human", "{input}"),
    ]
)
context_chain = RunnableLambda(add_reflexio_context) | context_prompt | llm

response = context_chain.invoke({"input": "I'm looking for a laptop for software development"})
show_success("Context-enriched chain response")
rprint(f"[bold]Agent:[/bold] {response.content[:300]}")

## 4. Always-On Context with `ReflexioChatModel`

`ReflexioChatModel` is a **drop-in ChatModel wrapper** that automatically searches Reflexio before **every** LLM call — no chain restructuring needed.

It works everywhere a `BaseChatModel` is accepted:
- LCEL chains (`prompt | reflexio_llm | parser`)
- `create_react_agent(reflexio_llm, tools)` — every agent iteration gets enriched
- `AgentExecutor`, LangGraph nodes, or any other LangChain construct

The wrapper intercepts `_generate`, `_stream`, and `_agenerate`, extracts the user's latest message as a search query, fetches relevant context from Reflexio, and injects it as a `SystemMessage` before delegating to the underlying model.

### Using with agents

Since `ReflexioChatModel` IS a `BaseChatModel`, it works directly with `create_react_agent` and other agent constructors. Every LLM call the agent makes (initial response, after tool results, etc.) automatically gets Reflexio context.

In [ ]:
from reflexio.integrations.langchain import ReflexioChatModel

# Wrap any chat model — every call now gets Reflexio context automatically
reflexio_llm = ReflexioChatModel(
    llm=llm,
    client=client,
    agent_version="v1.0",
    user_id=f"customer_{RUN_ID}",
    top_k=5,
)

In [ ]:
# Use the wrapped model with an agent — every LLM call gets Reflexio context
always_on_agent = create_react_agent(reflexio_llm, [product_search, order_lookup])

# Combine with callback handler to also capture conversations for learning
handler = ReflexioCallbackHandler(
    client=client,
    user_id=f"customer_{RUN_ID}",
    agent_version="v1.0",
    source="langchain_notebook",
)

result = always_on_agent.invoke(
    {"messages": [("human", "I'm looking for a laptop for software development. What do you have?")]},
    config={"callbacks": [handler]},
)
show_success("Always-on agent response (should now proactively ask about existing orders)")
rprint(f"[bold]Agent:[/bold] {result['messages'][-1].content[:400]}")

## 5. Agent Self-Lookup with `ReflexioSearchTool`

Give your agent a tool to query Reflexio mid-conversation. The agent decides when to look up past learnings — useful for complex queries where the agent is not sure how to handle a situation.

In [ ]:
from reflexio.integrations.langchain import ReflexioSearchTool

# Create the Reflexio search tool
reflexio_tool = ReflexioSearchTool(
    client=client,
    agent_version="v1.0",
    user_id=f"customer_{RUN_ID}",
)

# Combine with other tools
tools = [product_search, order_lookup, reflexio_tool]

In [ ]:
# Create a ReAct agent that can search Reflexio for guidance
self_improving_agent = create_react_agent(llm, tools)

handler = ReflexioCallbackHandler(
    client=client,
    user_id=f"customer_{RUN_ID}",
    agent_version="v1.0",
    source="langchain_notebook",
)
result = self_improving_agent.invoke(
    {"messages": [("human", "I need help choosing a laptop. Give me the full picture.")]},
    config={"callbacks": [handler]},
)
show_success("Self-lookup agent response")
rprint(f"[bold]Agent:[/bold] {result['messages'][-1].content[:400]}")

## 6. RAG-Style Retrieval with `ReflexioRetriever`

The retriever returns Reflexio results as LangChain `Document` objects, enabling standard RAG patterns like `create_retrieval_chain`.

In [ ]:
from reflexio.integrations.langchain import ReflexioRetriever

# Create retriever — searches across playbooks and profiles
retriever = ReflexioRetriever(
    client=client,
    agent_version="v1.0",
    search_type="unified",  # "unified", "playbooks", or "profiles"
    top_k=5,
)

# Retrieve relevant documents
docs = retriever.invoke("product recommendations")

display(Markdown(f"### Retrieved {len(docs)} Document(s)"))
for doc in docs:
    rprint(f"[bold][{doc.metadata.get('type', 'unknown')}][/bold] {doc.page_content[:150]}")
    rprint(f"  [dim]Metadata: {doc.metadata}[/dim]\n")

### Using the retriever in a RAG chain

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    """Format retrieved documents into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an Acme Electronics support agent. Use the following context from past interactions to guide your response:\n\n{context}",
        ),
        ("human", "{question}"),
    ]
)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("How should I help a customer looking for a laptop?")
show_success("RAG chain response")
rprint(f"[bold]Agent:[/bold] {answer[:400]}")

## 7. The Self-Improvement Loop

Here is how the full loop works over time:

```
+-------------------------------------------------------------------+
|                    Self-Improvement Loop                           |
|                                                                   |
|  1. Agent handles conversations                                   |
|     +-- ReflexioCallbackHandler publishes interactions             |
|                                                                   |
|  2. Reflexio processes in the background                          |
|     +-- Extracts user profiles (preferences, context)             |
|     +-- Extracts user playbooks (what went well/poorly)           |
|     +-- Aggregates playbooks into clusters                        |
|     +-- Generates actionable skills                               |
|                                                                   |
|  3. Next conversation starts                                      |
|     +-- ReflexioChatModel injects per-query context automatically |
|     +-- ReflexioSearchTool provides on-demand guidance            |
|     +-- Agent responds better based on learned behavior           |
|                                                                   |
|  4. Repeat -- agent improves with every conversation              |
+-------------------------------------------------------------------+
```

## Summary

| Component | Use Case | LangChain Dependency |
|---|---|---|
| `ReflexioCallbackHandler` | Capture conversations automatically | Yes |
| `get_reflexio_context` | Per-query context injection | No |
| `ReflexioChatModel` | Always-on per-query context (drop-in wrapper) | Yes |
| `ReflexioSearchTool` | Agent self-lookup mid-conversation | Yes |
| `ReflexioRetriever` | RAG-style document retrieval | Yes |

**Minimal integration** (covers most use cases):
```python
handler = ReflexioCallbackHandler(client, user_id="...", agent_version="v1")
agent.invoke({"messages": [...]}, config={"callbacks": [handler]})
```

**Always-on Reflexio context** (recommended for agents):
```python
reflexio_llm = ReflexioChatModel(llm=llm, client=client, agent_version="v1", user_id="...")
agent = create_react_agent(reflexio_llm, tools)
handler = ReflexioCallbackHandler(client, user_id="...", agent_version="v1")
agent.invoke({"messages": [...]}, config={"callbacks": [handler]})
```

### Next Steps

| Notebook | What you'll learn |
|----------|-------------------|
| [06 -- Concurrent Sessions](05_concurrent_sessions.ipynb) | Multi-user load testing and data isolation |
| [07 -- Simulation](06_real_world_simulation.ipynb) | Watch Reflexio learn from real conversations |

## Cleanup (Optional)

Remove the demo data created by this notebook.

In [ ]:
try:
    client.delete_all_interactions()
    show_success("Interactions cleaned up")
except Exception:
    pass  # Safe to skip

In [ ]:
try:
    client.delete_all_profiles()
    show_success("Profiles cleaned up")
except Exception:
    pass  # Safe to skip

show_success("All demo data removed")